In [ ]:
# CSBS / EDI REDCap scoring audit - single self-contained Colab cell
# Credentials are read from Colab Secrets (NANO and NANODD); they are never printed or embedded.

import getpass
import html
import math
import os
import re
from decimal import Decimal, ROUND_HALF_UP

import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, display


API_URL = "https://redcap.research.sc.edu/api/"
TARGET_FORMS = {
    "NANO": ["csbs_caregiver", "emotion_dysregulation_inventory_young_child"],
    "NANODD": ["csbs_bs"],
}

CG_SCALE_PERCENTILE = {
    17: 99, 16: 98, 15: 95, 14: 91, 13: 84, 12: 75, 11: 63,
    10: 50, 9: 37, 8: 25, 7: 16, 6: 9, 5: 5, 4: 2, 3: 1,
}
CG_TOTAL_PERCENTILE = {
    135: 99, 134: 99, 133: 99, 132: 98, 131: 98, 130: 98,
    129: 97, 128: 97, 127: 96, 126: 96, 125: 95, 124: 95,
    123: 94, 122: 93, 121: 92, 120: 91, 119: 90, 118: 89,
    117: 87, 116: 86, 115: 84, 114: 83, 113: 81, 112: 79,
    111: 77, 110: 75, 109: 73, 108: 70, 107: 68, 106: 66,
    105: 63, 104: 61, 103: 58, 102: 55, 101: 53, 100: 50,
    99: 47, 98: 45, 97: 42, 96: 40, 95: 37, 94: 35, 93: 32,
    92: 30, 91: 27, 90: 25, 89: 23, 88: 21, 87: 19, 86: 18,
    85: 16, 84: 14, 83: 13, 82: 12, 81: 10, 80: 9, 79: 8,
    78: 7, 77: 6, 76: 6, 75: 5, 74: 4, 73: 4, 72: 3, 71: 3,
    70: 2, 69: 2, 68: 2, 67: 1, 66: 1, 65: 1,
}

# Supplied CSBS Caregiver norms, age 23-24 months (attached PDF book pp. 154-160).
# Each tuple is (raw minimum, raw maximum, standard score).
CG_NORMS_23_24 = {
    "csbs_emotionandeyegaze": [(16,16,17),(15,15,14),(14,14,11),(13,13,9),(12,12,7),(11,11,5),(10,10,4),(0,9,3)],
    "csbs_communication": [(20,20,17),(19,19,12),(18,18,10),(16,17,9),(15,15,8),(12,14,6),(10,11,5),(9,9,4),(0,8,3)],
    "csbs_gestures": [(12,12,17),(11,11,9),(10,10,6),(9,9,5),(7,8,4),(0,6,3)],
    "csbs_sounds": [(16,16,17),(15,15,11),(14,14,10),(12,13,9),(11,11,8),(9,10,7),(7,8,6),(6,6,5),(0,5,3)],
    "csbs_words": [(24,24,17),(23,23,12),(20,22,11),(17,19,10),(13,16,9),(10,12,8),(8,9,7),(4,7,6),(3,3,5),(2,2,4),(0,1,3)],
    "csbs_understanding": [(24,24,17),(23,23,10),(22,22,9),(20,21,8),(15,19,7),(12,14,6),(7,11,5),(6,6,4),(0,5,3)],
    "csbs_objectuse": [(27,27,17),(25,26,12),(23,24,11),(22,22,10),(20,21,9),(18,19,8),(16,17,7),(14,15,6),(13,13,5),(10,12,4),(0,9,3)],
    "csbs_socialcomposite": [(48,48,17),(47,47,15),(46,46,14),(45,45,12),(43,44,11),(42,42,10),(40,41,9),(39,39,8),(38,38,7),(35,37,6),(34,34,5),(27,33,4),(0,26,3)],
    "csbs_speechcomposite": [(40,40,17),(39,39,13),(37,38,12),(33,36,11),(30,32,10),(25,29,9),(21,24,8),(16,20,7),(12,15,6),(10,11,5),(9,9,4),(0,8,3)],
    "csbs_symboliccomposite": [(51,51,17),(50,50,13),(49,49,12),(46,48,11),(44,45,10),(41,43,9),(36,40,8),(34,35,7),(27,33,6),(24,26,5),(17,23,4),(0,16,3)],
    "cbscg_totalscore": [(139,139,135),(138,138,131),(136,137,125),(135,135,124),(134,134,118),(133,133,115),(132,132,114),(130,131,112),(129,129,111),(128,128,110),(127,127,108),(126,126,106),(125,125,105),(124,124,104),(122,123,103),(121,121,102),(120,120,101),(119,119,100),(118,118,99),(117,117,98),(116,116,97),(115,115,96),(114,114,95),(112,113,94),(110,111,93),(109,109,92),(106,108,91),(104,105,90),(102,103,89),(100,101,88),(99,99,87),(98,98,86),(96,97,85),(94,95,84),(93,93,83),(88,92,82),(87,87,81),(84,86,80),(82,83,79),(80,81,78),(79,79,77),(77,78,76),(75,76,75),(74,74,74),(73,73,73),(72,72,72),(71,71,71),(61,70,70),(59,60,69),(56,58,68),(53,55,67),(50,52,66),(0,49,65)],
}
CG_RAW_MAX = {
    "csbs_emotionandeyegaze": 16, "csbs_communication": 20, "csbs_gestures": 12,
    "csbs_sounds": 16, "csbs_words": 24, "csbs_understanding": 24,
    "csbs_objectuse": 27, "csbs_socialcomposite": 48, "csbs_speechcomposite": 40,
    "csbs_symboliccomposite": 51, "cbscg_totalscore": 139,
}
def validate_cg_norm_domains():
    for field, maximum in CG_RAW_MAX.items():
        covered = [raw for low, high, _ in CG_NORMS_23_24[field] for raw in range(low, high + 1)]
        if sorted(covered) != list(range(maximum + 1)) or len(covered) != len(set(covered)):
            raise ValueError(f"Incomplete or overlapping 23-24 month norm table: {field}")


validate_cg_norm_domains()

CG_NORM_FIELDS = {
    "csbs_emotionandeyegaze": ("Emotion & eye gaze", "csbs_emotionss", "csbs_emotionandeyegazeper"),
    "csbs_communication": ("Communication", "csbs_communicationss", "csbs_communicationper"),
    "csbs_gestures": ("Gestures", "csbs_gesturesss", "csbs_gesturesper"),
    "csbs_sounds": ("Sounds", "csbs_soundsss", "csbs_soundsper"),
    "csbs_words": ("Words", "csbs_wordsss", "csbs_wordsper"),
    "csbs_understanding": ("Understanding", "csbs_understandingss", "csbs_understandingper"),
    "csbs_objectuse": ("Object use", "csbs_objectusess", "csbs_objectuseper"),
    "csbs_socialcomposite": ("Social composite", "csbs_socialcompositess", "csbs_socialcompositeper"),
    "csbs_speechcomposite": ("Speech composite", "csbs_speechcompositess", "csbs_speechcompositeper"),
    "csbs_symboliccomposite": ("Symbolic composite", "csbs_symboliccompositess", "csbs_symboliccompositeper"),
    "cbscg_totalscore": ("Total", "csbs_totalss", "csbs_totalper"),
}

CG_RAW_LABELS = {field: label for field, (label, _, _) in CG_NORM_FIELDS.items()}
BS_RAW_LABELS = {
    "csbsbs_emotionraw": "Emotion & eye gaze",
    "csbsbs_comraw": "Communication",
    "csbsbs_gesraw": "Gestures",
    "csbsbs_soundsraw": "Sounds",
    "csbsbs_wordsraw": "Words",
    "csbsbs_underraw": "Understanding",
    "csbsbs_objectraw": "Object use",
    "csbsbs_socialcompositecalc": "Social composite",
    "csbsbs_speechcompositecalc": "Speech composite",
    "csbsbs_symboliccompositecalc": "Symbolic composite",
    "csbsbs_totalrawcalc": "Total",
}


def redcap_post(token, content, **params):
    data = [("token", token), ("content", content), ("format", "json"), ("returnFormat", "json")]
    for key, value in params.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            data.extend((f"{key}[{index}]", str(item)) for index, item in enumerate(value))
        else:
            data.append((key, str(value)))
    response = requests.post(API_URL, data=data, timeout=180)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(payload["error"])
    return payload


def secret_value(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return str(value).strip() if value else ""
    except Exception:
        return ""


def get_project_tokens():
    secret_names = {
        "NANO": ("NANO", "REDCAP_NANO_TOKEN", "NANO_REDCAP_TOKEN", "NANO_TOKEN"),
        "NANODD": ("NANODD", "REDCAP_NANODD_TOKEN", "NANODD_REDCAP_TOKEN", "NANODD_TOKEN"),
    }
    resolved = {}
    for study, names in secret_names.items():
        token = next(
            (value for name in names if (value := secret_value(name) or str(os.getenv(name, "")).strip())),
            "",
        )
        if not token:
            token = getpass.getpass(f"{study} REDCap API token (hidden): ").strip()
        project = redcap_post(token, "project")
        project = project[0] if isinstance(project, list) and project else project
        title = str(project.get("project_title", ""))
        is_nanodd = "Double Data" in title or "Lab Assessment" in title
        if (study == "NANODD") != is_nanodd:
            raise RuntimeError(f"Colab Secret {study} points to the wrong REDCap project: {title}")
        resolved[study] = {"token": token, "project": project}
    return resolved


def fetch_bundle(study, token_info):
    token = token_info["token"]
    metadata = redcap_post(token, "metadata")
    record_id_field = str(metadata[0]["field_name"])
    forms = TARGET_FORMS[study]
    records = redcap_post(
        token,
        "record",
        type="flat",
        forms=forms,
        fields=[record_id_field, "demo_dob", "demo_gacalc", "demo_gaweeks", "demo_gadays"],
        rawOrLabel="raw",
        rawOrLabelHeaders="raw",
    )
    return {
        "study": study,
        "project": token_info["project"],
        "record_id_field": record_id_field,
        "metadata": [row for row in metadata if row.get("form_name") in forms],
        "records": records,
    }


def numeric(value):
    if value in (None, ""):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def complete_values(row, fields):
    values = [numeric(row.get(field)) for field in fields]
    return values if all(value is not None for value in values) else None


def ceil_sum(row, fields, transforms=None):
    values = complete_values(row, fields)
    if values is None:
        return None
    transforms = transforms or {}
    adjusted = [transforms.get(field, lambda x: x)(value) for field, value in zip(fields, values)]
    return float(math.ceil(sum(adjusted) - 1e-12))


def half_up(value):
    return float(Decimal(str(value)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


def cg_expected(row):
    emotion_fields = [f"csbscg{i}" for i in range(1, 9)]
    communication_fields = [f"csbscg{i}" for i in range(9, 19)]
    gestures_fields = ["csbscg19"] + [f"csbscg20_{i}" for i in range(1, 11)]
    sounds_fields = ["csbscg21", "csbscg22"] + [f"csbscg23_{i}" for i in range(1, 11)] + ["csbscg24"]
    words_fields = ["csbscg25"] + [f"csbscg26_{i}" for i in range(1, 37)] + ["csbscg27", "csbscg28"]
    understanding_fields = ["csbscg29", "csbscg30", "csbscg31"] + [f"csbscg32_{i}" for i in range(1, 37)]
    object_fields = ["csbscg33", "csbscg34"] + [f"csbscg35_{i}" for i in range(1, 11)] + ["csbscg36"] + [f"csbscg37_{i}" for i in range(1, 9)] + ["csbscg38", "csbscg39"] + [f"csbscg40_{i}" for i in range(1, 11)] + [f"csbscg41_{i}" for i in range(1, 7)]
    half_fields = {field: (lambda x: x / 2) for field in words_fields + understanding_fields + object_fields if re.match(r"csbscg(26|32|35|37|40|41)_", field)}

    emotion = ceil_sum(row, emotion_fields, {"csbscg5": lambda x: 2 - x})
    communication = ceil_sum(row, communication_fields)
    gestures = ceil_sum(row, gestures_fields)
    sounds = ceil_sum(row, sounds_fields)
    words = ceil_sum(row, words_fields, half_fields)
    understanding = ceil_sum(row, understanding_fields, half_fields)
    object_use = ceil_sum(row, object_fields, half_fields)
    values = [emotion, communication, gestures, sounds, words, understanding, object_use]
    if any(value is None for value in values):
        return None
    social, speech, symbolic = emotion + communication + gestures, sounds + words, understanding + object_use
    return {
        "csbs_emotionandeyegaze": emotion, "csbs_communication": communication,
        "csbs_gestures": gestures, "csbs_sounds": sounds, "csbs_words": words,
        "csbs_understanding": understanding, "csbs_objectuse": object_use,
        "csbs_socialcomposite": social, "csbs_speechcomposite": speech,
        "csbs_symboliccomposite": symbolic, "cbscg_totalscore": social + speech + symbolic,
    }


def bs_expected(row):
    required = [f"csbsbs_scale{i}" for i in range(1, 16)] + ["csbsbs_scale16_1", "csbsbs_scale16_2", "csbsbs_scale16_3", "csbsbs_scale17", "csbsbs_scale18", "csbsbs_scale19", "csbsbs_scale20"]
    if complete_values(row, required) is None:
        return None
    g = lambda name: numeric(row.get(name))
    emotion = half_up(g("csbsbs_scale1") + g("csbsbs_scale2") + 3 * g("csbsbs_scale3"))
    communication = half_up(g("csbsbs_scale4") / 3 + g("csbsbs_scale5") + g("csbsbs_scale6") + g("csbsbs_scale7"))
    gestures = half_up(2 * g("csbsbs_scale8") + g("csbsbs_scale9"))
    sounds = half_up(g("csbsbs_scale10") + 2 * g("csbsbs_scale11"))
    words = half_up(g("csbsbs_scale12") + g("csbsbs_scale13") / 2 + g("csbsbs_scale14") + g("csbsbs_scale15"))
    understanding = half_up(3 * (g("csbsbs_scale16_1") + g("csbsbs_scale16_2") + g("csbsbs_scale16_3")))
    object_use = half_up(sum(g(f"csbsbs_scale{i}") for i in range(17, 21)))
    return {
        "csbsbs_emotionraw": emotion, "csbsbs_comraw": communication,
        "csbsbs_gesraw": gestures, "csbsbs_soundsraw": sounds,
        "csbsbs_wordsraw": words, "csbsbs_underraw": understanding,
        "csbsbs_objectraw": object_use,
        "csbsbs_socialcompositecalc": emotion + communication + gestures,
        "csbsbs_speechcompositecalc": sounds + words,
        "csbsbs_symboliccompositecalc": understanding + object_use,
        "csbsbs_totalrawcalc": emotion + communication + gestures + sounds + words + understanding + object_use,
    }


def cg_norm(raw_field, raw_value):
    if raw_value is None:
        return None, None
    raw_value = int(round(raw_value))
    for low, high, standard in CG_NORMS_23_24[raw_field]:
        if low <= raw_value <= high:
            percentile = CG_TOTAL_PERCENTILE[standard] if raw_field == "cbscg_totalscore" else CG_SCALE_PERCENTILE[standard]
            return standard, percentile
    return None, None


def parse_crosswalk(annotation, raw_field):
    pattern = re.compile(rf"\[{re.escape(raw_field)}\]\s*=\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)", re.I)
    return {int(float(raw)): float(score) for raw, score in pattern.findall(annotation or "")}


audit_rows = []


def add_check(study, instrument, record_id, event, group, score, field, actual, expected, primary, issue="", tolerance=1e-9):
    actual_number = numeric(actual)
    if expected is None:
        status, difference = "Unverified", np.nan
    elif actual_number is None:
        status, difference = "Missing", np.nan
    else:
        difference = actual_number - float(expected)
        status = "Match" if abs(difference) <= tolerance else "Review"
    audit_rows.append({
        "Study": study, "Instrument": instrument, "Record ID": str(record_id or ""),
        "Event / Visit": str(event or ""), "Group": group, "Score": score,
        "REDCap Field": field, "REDCap Value": actual_number, "Expected Value": expected,
        "Difference": difference, "Absolute Difference": abs(difference) if pd.notna(difference) else np.nan,
        "Status": status, "Primary": bool(primary), "Issue": issue if status != "Match" else "",
    })


token_info = get_project_tokens()
bundles = {study: fetch_bundle(study, info) for study, info in token_info.items()}

# --- CSBS Caregiver: worksheet-weighted raw scores and 23-24 month normed fields ---
nano = bundles["NANO"]
nano_id = nano["record_id_field"]
for row in nano["records"]:
    expected = cg_expected(row)
    if expected is None:
        continue
    record_id, event = row.get(nano_id, ""), row.get("redcap_event_name", "")
    for field, expected_value in expected.items():
        actual = row.get(field, "")
        if field == "csbs_symboliccomposite":
            issue = "REDCap rounds the combined Symbolic item terms once; the scoring worksheet requires summing the separately rounded Understanding and Object Use cluster scores."
        elif field == "cbscg_totalscore":
            issue = "REDCap recomputes rounded item groups; the scoring worksheet requires summing the three composite weighted raw scores, so a Symbolic rounding difference propagates to Total."
        else:
            issue = "Stored REDCap calculation does not equal the current worksheet-weighted item values; review possible stale calculation or changed source responses."
        add_check("NANO", "CSBS Caregiver", record_id, event, "CG raw", f"{CG_RAW_LABELS[field]} raw", field, actual, expected_value, True, issue)

    if event != "24_months_arm_1":
        continue
    for raw_field, (scale, ss_field, percentile_field) in CG_NORM_FIELDS.items():
        recorded_raw = numeric(row.get(raw_field))
        independent_raw = expected[raw_field]
        expected_ss, expected_percentile = cg_norm(raw_field, independent_raw)
        for kind, field, norm_expected in (
            ("Standard score", ss_field, expected_ss),
            ("Percentile rank", percentile_field, expected_percentile),
        ):
            if row.get(field, "") in (None, ""):
                continue
            issue = "Entered REDCap score does not equal the supplied 23-24 month norm lookup applied to the independently recalculated raw score."
            if recorded_raw != independent_raw:
                issue += " The stored raw score also differs from the worksheet-based calculation."
            add_check("NANO", "CSBS Caregiver", record_id, event, "CG manual norm", f"{scale} {kind}", field, row.get(field), norm_expected, True, issue)

# --- CSBS BS: independent worksheet formulas from the supplied scoring sheets ---
nanodd = bundles["NANODD"]
nanodd_id = nanodd["record_id_field"]
for row in nanodd["records"]:
    expected = bs_expected(row)
    if expected is None:
        continue
    record_id, event = row.get(nanodd_id, ""), row.get("redcap_event_name", "")
    for field, expected_value in expected.items():
        add_check("NANODD", "CSBS BS", record_id, event, "BS raw", f"{BS_RAW_LABELS[field]} raw", field, row.get(field), expected_value, True, "REDCap raw value differs from an independent evaluation of the instrument formula.")

bs_meta = {row["field_name"]: row for row in nanodd["metadata"]}
bs_manual_fields = [
    name for name, meta in bs_meta.items()
    if meta.get("field_type") in ("text", "notes")
    and re.search(r"(?i)(standard score|percentile rank|age equivalent)", meta.get("field_label", ""))
]
for row in nanodd["records"]:
    for field in bs_manual_fields:
        if row.get(field, "") in (None, ""):
            continue
        add_check("NANODD", "CSBS BS", row.get(nanodd_id, ""), row.get("redcap_event_name", ""), "BS manual norms", re.sub(r"\s+", " ", bs_meta[field].get("field_label", field)).strip(), field, row.get(field), None, False, "The supplied PDF does not include the CSBS BS Appendix C raw-to-standard/percentile/age-equivalent crosswalks.")

# --- EDI-YC: hidden API fields, item sums, and metadata-parsed raw-to-T crosswalks ---
edi_meta = {row["field_name"]: row for row in nano["metadata"] if row.get("form_name") == "emotion_dysregulation_inventory_young_child"}
edi_defs = {
    "Reactivity Short Form": {"raw":"edi_yc_rsf", "clin_sources":["edi_yc_rsf_clin"], "gen_sources":["edi_yc_rsf_gen"], "clin_display":"edi_yc_rsf_tdisp", "gen_display":"edi_yc_rsf_gdisp"},
    "Dysphoria Index": {"raw":"edi_yc_di", "clin_sources":["edi_yc_di_clin"], "gen_sources":["edi_yc_di_gen"], "clin_display":"edi_yc_di_tdisp", "gen_display":"edi_yc_di_gdisp"},
    "Full Reactivity Index": {"raw":"edi_yc_fri", "clin_sources":["edi_yc_df1_clin", "edi_yc_df2_clin"], "gen_sources":["edi_yc_df1_gen", "edi_yc_df2_gen"], "clin_display":"edi_yc_df_tdisp", "gen_display":"edi_yc_df_gdisp"},
}
for scale, definition in edi_defs.items():
    raw_field = definition["raw"]
    item_fields = list(dict.fromkeys(re.findall(r"\[(edi_yc_q\d+)\]", edi_meta[raw_field]["select_choices_or_calculations"])))
    clinical, general = {}, {}
    for source in definition["clin_sources"]:
        clinical.update(parse_crosswalk(edi_meta[source].get("field_annotation", ""), raw_field))
    for source in definition["gen_sources"]:
        general.update(parse_crosswalk(edi_meta[source].get("field_annotation", ""), raw_field))

    for row in nano["records"]:
        values = complete_values(row, item_fields)
        if values is None:
            continue
        record_id, event = row.get(nano_id, ""), row.get("redcap_event_name", "")
        raw_expected = float(sum(values))
        add_check("NANO", "EDI-YC", record_id, event, "EDI raw", f"{scale} raw", raw_field, row.get(raw_field), raw_expected, True, "EDI raw score does not equal the sum of its required items.")
        raw_integer = int(raw_expected)
        add_check("NANO", "EDI-YC", record_id, event, "EDI T", f"{scale} clinical T", definition["clin_display"], row.get(definition["clin_display"]), clinical.get(raw_integer), True, "Clinical T-score differs from the metadata crosswalk.", 0.0005)
        add_check("NANO", "EDI-YC", record_id, event, "EDI T", f"{scale} general T", definition["gen_display"], row.get(definition["gen_display"]), general.get(raw_integer), True, "General T-score differs from the metadata crosswalk.", 0.0005)

config_issues = []
if "[edi_yc_rsf_gen]" in edi_meta["edi_yc_di_gdisp"].get("field_annotation", ""):
    config_issues.append("edi_yc_di_gdisp checks whether edi_yc_rsf_gen is blank; it should check edi_yc_di_gen.")

audit = pd.DataFrame(audit_rows)


def summarize(label, frame, override_status=None):
    compared = frame[frame["Status"].isin(["Match", "Review"])]
    mismatches = compared[compared["Status"] == "Review"]
    unverified = frame[frame["Status"].isin(["Unverified", "Missing"])]
    deltas = pd.to_numeric(compared["Difference"], errors="coerce").dropna()
    match_rate = 100 * (len(compared) - len(mismatches)) / len(compared) if len(compared) else np.nan
    if override_status:
        decision = override_status
    elif len(compared) == 0 and len(unverified):
        decision = "Needs source"
    elif len(mismatches) == 0 and len(unverified) == 0:
        decision = "Verified"
    elif len(mismatches) == 0:
        decision = "Coverage gap"
    elif match_rate >= 99:
        decision = "Watch"
    else:
        decision = "Review"
    record_events = frame[["Record ID", "Event / Visit"]].drop_duplicates() if len(frame) else pd.DataFrame()
    return {
        ("Scope", "Audit layer"): label,
        ("Coverage", "Record-events"): len(record_events),
        ("Coverage", "Compared"): len(compared),
        ("Coverage", "Not verified"): len(unverified),
        ("Accuracy", "Differences"): len(mismatches),
        ("Accuracy", "Match rate"): match_rate,
        ("Difference variability", "MAE"): deltas.abs().mean() if len(deltas) else np.nan,
        ("Difference variability", "SD(Δ)"): deltas.std(ddof=1) if len(deltas) > 1 else (0.0 if len(deltas) == 1 else np.nan),
        ("Difference variability", "Max |Δ|"): deltas.abs().max() if len(deltas) else np.nan,
        ("Decision", "Status"): decision,
    }


summary_rows = [
    summarize("CSBS Caregiver - REDCap auto raw vs worksheet logic", audit[audit["Group"] == "CG raw"]),
    summarize("CSBS Caregiver - REDCap-entered SS/percentile vs expected", audit[audit["Group"] == "CG manual norm"]),
    summarize("CSBS BS - REDCap auto raw vs worksheet logic", audit[audit["Group"] == "BS raw"]),
    summarize("CSBS BS - manual SS/percentile/age equivalent", audit[audit["Group"] == "BS manual norms"]),
    summarize("EDI-YC - REDCap auto raw vs item sum", audit[audit["Group"] == "EDI raw"]),
    summarize("EDI-YC - displayed clinical/general T vs crosswalk", audit[audit["Group"] == "EDI T"]),
]

if config_issues:
    summary_rows.append({
        ("Scope", "Audit layer"): "EDI-YC - metadata dependency logic",
        ("Coverage", "Record-events"): 0,
        ("Coverage", "Compared"): 1,
        ("Coverage", "Not verified"): 0,
        ("Accuracy", "Differences"): len(config_issues),
        ("Accuracy", "Match rate"): 0.0,
        ("Difference variability", "MAE"): np.nan,
        ("Difference variability", "SD(Δ)"): np.nan,
        ("Difference variability", "Max |Δ|"): np.nan,
        ("Decision", "Status"): "Fix metadata",
    })

summary = pd.DataFrame(summary_rows)
summary.columns = pd.MultiIndex.from_tuples(summary.columns)

primary = audit[audit["Primary"] & audit["Status"].isin(["Match", "Review"])]
primary_mismatches = primary[primary["Status"] == "Review"]
primary_rate = 100 * (len(primary) - len(primary_mismatches)) / len(primary) if len(primary) else np.nan
review_record_events = primary_mismatches[["Study", "Record ID", "Event / Visit"]].drop_duplicates()

display(HTML(f"""
<style>
.audit-wrap {{font-family:Arial,sans-serif;color:#17324d;max-width:1400px;margin:auto}}
.audit-title {{background:linear-gradient(90deg,#12344d,#1f5f7a);color:white;padding:18px 22px;border-radius:10px;margin:8px 0 14px}}
.audit-title h2 {{margin:0 0 5px;font-size:24px}} .audit-title p {{margin:0;color:#dceaf2}}
.audit-cards {{display:grid;grid-template-columns:repeat(4,minmax(160px,1fr));gap:12px;margin:12px 0 18px}}
.audit-card {{border:1px solid #d8e3ea;border-radius:9px;padding:14px;text-align:center;background:#f8fbfd}}
.audit-card b {{display:block;font-size:24px;color:#12344d;margin-bottom:4px}} .audit-card span {{font-size:12px;color:#587084}}
.audit-note {{border-left:5px solid #e0a100;background:#fff8e1;color:#3f3300;padding:11px 14px;margin:12px 0;border-radius:6px}}
</style>
<div class="audit-wrap">
  <div class="audit-title"><h2>REDCap CSBS / EDI Scoring Accuracy Audit</h2><p>Independent item recomputation, norm-table validation, and record-level discrepancy metadata</p></div>
  <div class="audit-cards">
    <div class="audit-card"><b>{len(primary):,}</b><span>Primary comparisons</span></div>
    <div class="audit-card"><b>{primary_rate:.2f}%</b><span>Primary match rate</span></div>
    <div class="audit-card"><b>{len(review_record_events):,}</b><span>Record-events needing review</span></div>
    <div class="audit-card"><b>{len(config_issues)}</b><span>Metadata dependency defects</span></div>
  </div>
</div>
"""))


def summary_row_style(row):
    status = row[("Decision", "Status")]
    color = {"Verified":"#e7f6ec", "Watch":"#fff8df", "Review":"#fdeaea", "Coverage gap":"#fff8df", "Needs source":"#eef2f5", "Fix metadata":"#fdeaea"}.get(status, "white")
    return [f"background-color:{color}" for _ in row]


summary_style = (
    summary.style.hide(axis="index")
    .format({
        ("Accuracy", "Match rate"): lambda value: "-" if pd.isna(value) else f"{value:.2f}%",
        ("Difference variability", "MAE"): lambda value: "-" if pd.isna(value) else f"{value:.3f}",
        ("Difference variability", "SD(Δ)"): lambda value: "-" if pd.isna(value) else f"{value:.3f}",
        ("Difference variability", "Max |Δ|"): lambda value: "-" if pd.isna(value) else f"{value:.3f}",
    })
    .apply(summary_row_style, axis=1)
    .set_properties(**{"text-align":"center", "color":"#17212b", "border":"1px solid #d9e2e8", "padding":"7px"})
    .set_properties(subset=[("Scope", "Audit layer")], **{"text-align":"left", "font-weight":"600"})
    .set_table_styles([
        {"selector":"th", "props":[("background-color","#12344d"),("color","white"),("text-align","center"),("padding","8px"),("border","1px solid #d9e2e8")]},
        {"selector":"caption", "props":[("caption-side","top"),("text-align","left"),("font-weight","700"),("font-size","17px"),("color","#12344d"),("padding","8px 0")]},
    ])
    .set_caption("Simplified stakeholder comparison table")
)
display(summary_style)

if config_issues:
    config_html = "<br>".join(f"• {html.escape(issue)}" for issue in config_issues)
    display(HTML(f"<div class='audit-note'><b>Configuration defect:</b><br>{config_html}<br><small>Complete EDI records can still match while partial future records remain at risk.</small></div>"))

formula_legend = pd.DataFrame([
    ["Δ", "REDCap value - independent expected value", "All numeric comparisons"],
    ["Match rate", "100 × matching comparisons / comparisons made", "Accuracy"],
    ["MAE", "mean(|Δ|); zeros included", "Typical error size"],
    ["SD(Δ)", "sample standard deviation of Δ", "Error variability"],
    ["CSBS Caregiver raw", "ceil(weighted item sum); item 5 reversed; inventory groups 26/32/35/37/40/41 weighted ÷2; composite = cluster sum; total = composite sum", "All complete Caregiver record-events"],
    ["Caregiver hierarchy check", "expected Symbolic = Understanding + Object Use; expected Total = Social + Speech + Symbolic. Current REDCap instead rounds combined underlying item terms, which can differ by 1", "Explains systematic Symbolic/Total discrepancies"],
    ["CSBS Caregiver norm", "independently calculated raw → 23-24 month standard-score lookup → published percentile", "24-month event only; supplied PDF pp. 154-160"],
    ["CSBS BS raw", "E+EG=s1+s2+3s3; Communication=round(s4/3+s5+s6+s7); Gestures=round(2s8+s9); Sounds=round(s10+2s11); Words=round(s12+s13/2+s14+s15); Understanding=round(3(s16a+s16b+s16c)); Object Use=round(s17+s18+s19+s20); composite/total = sums", "All complete BS record-events; supplied worksheets"],
    ["EDI-YC", "Raw = required-item sum; T = exact raw-to-T pair parsed from hidden @CALCTEXT metadata; compare API-exported *_tdisp/*_gdisp", "Complete EDI record-events"],
], columns=["Code / layer", "Minimal formula or rule", "Applies to"])
display(
    formula_legend.style.hide(axis="index")
    .set_properties(**{"text-align":"left", "vertical-align":"top", "color":"#17212b", "background-color":"#ffffff", "border":"1px solid #d9e2e8", "padding":"7px"})
    .set_table_styles([
        {"selector":"th", "props":[("background-color","#dceaf2"),("color","#12344d"),("text-align","center"),("padding","8px")]},
        {"selector":"caption", "props":[("caption-side","top"),("text-align","left"),("font-weight","700"),("font-size","17px"),("color","#12344d"),("padding","12px 0 6px")]},
    ])
    .set_caption("Legend and calculation rules")
)

display(HTML("<div class='audit-note'><b>Source boundary:</b> The Caregiver 23-24 month raw-to-standard/percentile tables are fully represented. The supplied PDF explains Behavior Sample scoring but does not contain Appendix C's BS standard-score, percentile, and age-equivalent crosswalks. Therefore BS raw scoring is audited from the supplied worksheets, while populated BS normed fields are explicitly not verified rather than inferred.</div>"))

discrepancies = audit[audit["Status"].isin(["Review", "Missing"])].copy()
discrepancies = discrepancies.sort_values(["Absolute Difference", "Study", "Instrument", "Record ID"], ascending=[False, True, True, True], na_position="last")
if len(discrepancies):
    metadata = discrepancies[["Study", "Record ID", "Event / Visit", "Instrument", "Group", "Score", "REDCap Field", "REDCap Value", "Expected Value", "Difference", "Issue"]].copy()
    metadata.columns = pd.MultiIndex.from_tuples([
        ("Record", "Study"), ("Record", "ID"), ("Record", "Event / visit"),
        ("Score", "Instrument"), ("Score", "Check"), ("Score", "Measure"), ("Score", "REDCap field"),
        ("Comparison", "REDCap"), ("Comparison", "Expected"), ("Comparison", "Δ"),
        ("Review", "Reason"),
    ])
    metadata_style = (
        metadata.style.hide(axis="index")
        .format({
            ("Comparison", "REDCap"): lambda value: "" if pd.isna(value) else f"{value:g}",
            ("Comparison", "Expected"): lambda value: "" if pd.isna(value) else f"{value:g}",
            ("Comparison", "Δ"): lambda value: "" if pd.isna(value) else f"{value:g}",
        })
        .set_properties(**{"text-align":"center", "vertical-align":"middle", "color":"#17212b", "border":"1px solid #ead3d3", "padding":"6px"})
        .set_properties(subset=[("Review", "Reason")], **{"text-align":"left", "min-width":"320px"})
        .set_table_styles([
            {"selector":"th", "props":[("background-color","#8b1e2d"),("color","white"),("text-align","center"),("padding","7px")]},
            {"selector":"td", "props":[("background-color","#fffafa"),("color","#17212b")]},
            {"selector":"caption", "props":[("caption-side","top"),("text-align","left"),("font-weight","700"),("font-size","17px"),("color","#8b1e2d"),("padding","12px 0 6px")]},
        ])
        .set_caption(f"Record-level discrepancy metadata ({len(metadata):,} score-level rows)")
    )
    display(HTML(f"<div style='max-height:650px;overflow:auto;border:1px solid #d9e2e8;border-radius:7px'>{metadata_style.to_html()}</div>"))
else:
    display(HTML("<div style='background:#e7f6ec;color:#153f24;padding:12px;border-radius:7px'><b>No record-level numeric discrepancies found.</b></div>"))

# Download-ready files in the Colab runtime.
summary_export = summary.copy()
summary_export.columns = [" | ".join(column) for column in summary_export.columns]
summary_export.to_csv("/content/csbs_edi_audit_summary.csv", index=False)
audit.to_csv("/content/csbs_edi_record_comparisons.csv", index=False)
discrepancies.to_csv("/content/csbs_edi_discrepancy_metadata.csv", index=False)
formula_legend.to_csv("/content/csbs_edi_formula_legend.csv", index=False)
print("\nSaved: /content/csbs_edi_audit_summary.csv")
print("Saved: /content/csbs_edi_record_comparisons.csv")
print("Saved: /content/csbs_edi_discrepancy_metadata.csv")
print("Saved: /content/csbs_edi_formula_legend.csv")


Code / layer,Minimal formula or rule,Applies to
Δ,REDCap value - independent expected value,All numeric comparisons
Match rate,100 × matching comparisons / comparisons made,Accuracy
MAE,mean(|Δ|); zeros included,Typical error size
SD(Δ),sample standard deviation of Δ,Error variability
CSBS Caregiver raw,ceil(weighted item sum); item 5 reversed; inventory groups 26/32/35/37/40/41 weighted ÷2; composite = cluster sum; total = composite sum,All complete Caregiver record-events
Caregiver hierarchy check,"expected Symbolic = Understanding + Object Use; expected Total = Social + Speech + Symbolic. Current REDCap instead rounds combined underlying item terms, which can differ by 1",Explains systematic Symbolic/Total discrepancies
CSBS Caregiver norm,independently calculated raw → 23-24 month standard-score lookup → published percentile,24-month event only; supplied PDF pp. 154-160
CSBS BS raw,E+EG=s1+s2+3s3; Communication=round(s4/3+s5+s6+s7); Gestures=round(2s8+s9); Sounds=round(s10+2s11); Words=round(s12+s13/2+s14+s15); Understanding=round(3(s16a+s16b+s16c)); Object Use=round(s17+s18+s19+s20); composite/total = sums,All complete BS record-events; supplied worksheets
EDI-YC,Raw = required-item sum; T = exact raw-to-T pair parsed from hidden @CALCTEXT metadata; compare API-exported *_tdisp/*_gdisp,Complete EDI record-events



Saved: /content/csbs_edi_audit_summary.csv
Saved: /content/csbs_edi_record_comparisons.csv
Saved: /content/csbs_edi_discrepancy_metadata.csv
Saved: /content/csbs_edi_formula_legend.csv


In [ ]:
# Coverage correction and final export pack
import json
import zipfile
from pathlib import Path

# Make this cell rerunnable without duplicating rows.
audit_rows = [r for r in audit_rows if r.get("Group") not in {"CG timepoint rule", "BS timepoint rule"}]
audit = pd.DataFrame(audit_rows)

# The first pass compared populated 24-month norm fields. Add the required blank fields as Missing.
existing = set(
    (str(r["Record ID"]), str(r["Event / Visit"]), str(r["REDCap Field"]))
    for r in audit_rows
    if r.get("Group") == "CG manual norm"
)
raw_only_events = {"6_months_arm_1", "9_months_arm_1", "12_months_arm_1"}
for row in nano["records"]:
    expected = cg_expected(row)
    if expected is None:
        continue
    record_id = str(row.get(nano_id, ""))
    event = str(row.get("redcap_event_name", ""))
    for raw_field, (scale, ss_field, percentile_field) in CG_NORM_FIELDS.items():
        expected_ss, expected_percentile = cg_norm(raw_field, expected[raw_field])
        if event == "24_months_arm_1":
            for kind, field, norm_expected in (
                ("Standard score", ss_field, expected_ss),
                ("Percentile rank", percentile_field, expected_percentile),
            ):
                key = (record_id, event, field)
                if key not in existing:
                    add_check(
                        "NANO", "CSBS Caregiver", record_id, event, "CG manual norm",
                        f"{scale} {kind}", field, row.get(field), norm_expected, True,
                        "Required 24-month score is blank; expected value is from the supplied 23-24 month norm table.",
                    )
                    existing.add(key)
        elif event in raw_only_events:
            for kind, field in (("Standard score", ss_field), ("Percentile rank", percentile_field)):
                if row.get(field, "") not in (None, ""):
                    actual_value = numeric(row.get(field))
                    audit_rows.append({
                        "Study": "NANO", "Instrument": "CSBS Caregiver", "Record ID": record_id,
                        "Event / Visit": event, "Group": "CG timepoint rule",
                        "Score": f"{scale} {kind}", "REDCap Field": field,
                        "REDCap Value": actual_value, "Expected Value": np.nan,
                        "Difference": np.nan, "Absolute Difference": np.nan,
                        "Status": "Review", "Primary": True,
                        "Issue": "Normed Caregiver scores should be entered only at 24 months; this earlier visit should contain raw scores only.",
                    })

# Enforce the unambiguous BS visit rule: 12-month visits should not have age-equivalent entries.
for row in nanodd["records"]:
    event = str(row.get("redcap_event_name", ""))
    if event != "12_months_arm_1":
        continue
    for field in bs_manual_fields:
        label = re.sub(r"\s+", " ", bs_meta[field].get("field_label", field)).strip()
        if "age equivalent" in label.lower() and row.get(field, "") not in (None, ""):
            audit_rows.append({
                "Study": "NANODD", "Instrument": "CSBS BS",
                "Record ID": str(row.get(nanodd_id, "")), "Event / Visit": event,
                "Group": "BS timepoint rule", "Score": label, "REDCap Field": field,
                "REDCap Value": numeric(row.get(field)), "Expected Value": np.nan,
                "Difference": np.nan, "Absolute Difference": np.nan,
                "Status": "Review", "Primary": True,
                "Issue": "Age equivalents are not expected at 12 months under the stated workflow.",
            })

audit = pd.DataFrame(audit_rows)

# Final summary, including coverage gaps and visit-rule violations.
summary_rows = [
    summarize("CSBS Caregiver - REDCap auto raw vs worksheet logic", audit[audit["Group"] == "CG raw"]),
    summarize("CSBS Caregiver - REDCap-entered SS/percentile vs expected", audit[audit["Group"] == "CG manual norm"]),
    summarize("CSBS Caregiver - visit/timepoint field rules", audit[audit["Group"] == "CG timepoint rule"]),
    summarize("CSBS BS - REDCap auto raw vs worksheet logic", audit[audit["Group"] == "BS raw"]),
    summarize("CSBS BS - manual SS/percentile/age equivalent", audit[audit["Group"] == "BS manual norms"]),
    summarize("CSBS BS - visit/timepoint field rules", audit[audit["Group"] == "BS timepoint rule"]),
    summarize("EDI-YC - REDCap auto raw vs item sum", audit[audit["Group"] == "EDI raw"]),
    summarize("EDI-YC - displayed clinical/general T vs crosswalk", audit[audit["Group"] == "EDI T"]),
]
if config_issues:
    summary_rows.append({
        ("Scope", "Audit layer"): "EDI-YC - metadata dependency logic",
        ("Coverage", "Record-events"): 0,
        ("Coverage", "Compared"): 1,
        ("Coverage", "Not verified"): 0,
        ("Accuracy", "Differences"): len(config_issues),
        ("Accuracy", "Match rate"): 0.0,
        ("Difference variability", "MAE"): np.nan,
        ("Difference variability", "SD(Δ)"): np.nan,
        ("Difference variability", "Max |Δ|"): np.nan,
        ("Decision", "Status"): "Fix metadata",
    })
summary = pd.DataFrame(summary_rows)
summary.columns = pd.MultiIndex.from_tuples(summary.columns)

# Count instrument rows with enough input data to score, by event.
coverage_rows = []
def add_coverage(study, instrument, records, complete_fn, complete_field):
    grouped = {}
    for row in records:
        event = str(row.get("redcap_event_name", ""))
        marked = str(row.get(complete_field, "")) in {"1", "2"}
        scoreable = complete_fn(row) is not None
        active = marked or scoreable
        if not active:
            continue
        bucket = grouped.setdefault(event, {"active": 0, "scoreable": 0, "marked_complete": 0})
        bucket["active"] += 1
        bucket["scoreable"] += int(scoreable)
        bucket["marked_complete"] += int(marked)
    for event, counts in sorted(grouped.items()):
        coverage_rows.append({
            "Study": study, "Instrument": instrument, "Event / Visit": event,
            "Active record-events": counts["active"],
            "Scoreable record-events": counts["scoreable"],
            "Marked complete": counts["marked_complete"],
            "Not scoreable": counts["active"] - counts["scoreable"],
        })

add_coverage("NANO", "CSBS Caregiver", nano["records"], cg_expected, "csbs_caregiver_complete")
add_coverage("NANODD", "CSBS BS", nanodd["records"], bs_expected, "csbs_bs_complete")

edi_item_fields = sorted({f for d in edi_defs.values() for f in re.findall(r"\[(edi_yc_q\d+)\]", edi_meta[d["raw"]]["select_choices_or_calculations"])})
def edi_complete(row):
    vals = complete_values(row, edi_item_fields)
    return vals if vals is not None else None
add_coverage("NANO", "EDI-YC", nano["records"], edi_complete, "emotion_dysregulation_inventory_young_child_complete")
coverage = pd.DataFrame(coverage_rows)

# Inventory the manual BS fields by event/category without claiming norm validity.
presence_rows = []
for row in nanodd["records"]:
    event = str(row.get("redcap_event_name", ""))
    record_id = str(row.get(nanodd_id, ""))
    for field in bs_manual_fields:
        if row.get(field, "") in (None, ""):
            continue
        label = re.sub(r"\s+", " ", bs_meta[field].get("field_label", field)).strip()
        lower = label.lower()
        category = "Age equivalent" if "age equivalent" in lower else ("Percentile rank" if "percentile" in lower else "Standard score")
        presence_rows.append({
            "Study": "NANODD", "Record ID": record_id, "Event / Visit": event,
            "Category": category, "Field": field, "Label": label, "Value": row.get(field),
        })
bs_presence = pd.DataFrame(presence_rows)

# Final discrepancy set includes numeric differences, missing required values, and timepoint-rule violations.
discrepancies = audit[audit["Status"].isin(["Review", "Missing"])].copy()
discrepancies = discrepancies.sort_values(
    ["Absolute Difference", "Study", "Instrument", "Record ID"],
    ascending=[False, True, True, True], na_position="last"
)

summary_export = summary.copy()
summary_export.columns = [" | ".join(column) for column in summary_export.columns]
outdir = Path("/content")
summary_export.to_csv(outdir / "csbs_edi_audit_summary.csv", index=False)
audit.to_csv(outdir / "csbs_edi_record_comparisons.csv", index=False)
discrepancies.to_csv(outdir / "csbs_edi_discrepancy_metadata.csv", index=False)
formula_legend.to_csv(outdir / "csbs_edi_formula_legend.csv", index=False)
coverage.to_csv(outdir / "csbs_edi_visit_coverage.csv", index=False)
bs_presence.to_csv(outdir / "csbs_bs_manual_field_presence.csv", index=False)

zip_path = outdir / "csbs_edi_audit_exports.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in [
        "csbs_edi_audit_summary.csv",
        "csbs_edi_record_comparisons.csv",
        "csbs_edi_discrepancy_metadata.csv",
        "csbs_edi_formula_legend.csv",
        "csbs_edi_visit_coverage.csv",
        "csbs_bs_manual_field_presence.csv",
    ]:
        zf.write(outdir / name, arcname=name)

primary_final = audit[audit["Primary"] & audit["Status"].isin(["Match", "Review", "Missing"])]
review_final = primary_final[primary_final["Status"].isin(["Review", "Missing"])]
review_events_final = review_final[["Study", "Record ID", "Event / Visit"]].drop_duplicates()
match_count = int((primary_final["Status"] == "Match").sum())
rate_final = 100 * match_count / len(primary_final) if len(primary_final) else np.nan

print(json.dumps({
    "primary_comparisons": int(len(primary_final)),
    "primary_match_rate_pct": round(float(rate_final), 4),
    "score_level_review_or_missing": int(len(review_final)),
    "record_events_needing_review": int(len(review_events_final)),
    "discrepancy_rows": int(len(discrepancies)),
    "missing_required_24m_norm_entries": int(((audit["Group"] == "CG manual norm") & (audit["Status"] == "Missing")).sum()),
    "cg_timepoint_rule_violations": int(((audit["Group"] == "CG timepoint rule") & (audit["Status"] == "Review")).sum()),
    "bs_12m_age_equivalent_violations": int(((audit["Group"] == "BS timepoint rule") & (audit["Status"] == "Review")).sum()),
    "metadata_dependency_defects": len(config_issues),
    "zip_path": str(zip_path),
}, indent=2))
display(summary.style.hide(axis="index"))
display(coverage.style.hide(axis="index"))


{
  "primary_comparisons": 18238,
  "primary_match_rate_pct": 72.8369,
  "score_level_review_or_missing": 4954,
  "record_events_needing_review": 318,
  "discrepancy_rows": 4954,
  "missing_required_24m_norm_entries": 52,
  "cg_timepoint_rule_violations": 4486,
  "bs_12m_age_equivalent_violations": 88,
  "metadata_dependency_defects": 1,
  "zip_path": "/content/csbs_edi_audit_exports.zip"
}


Study,Instrument,Event / Visit,Active record-events,Scoreable record-events,Marked complete,Not scoreable
NANO,CSBS Caregiver,12_months_arm_1,136,136,136,0
NANO,CSBS Caregiver,24_months_arm_1,82,82,82,0
NANO,CSBS Caregiver,6_months_arm_1,174,173,172,1
NANO,CSBS Caregiver,9_months_arm_1,154,152,154,2
NANODD,CSBS BS,12_months_arm_1,285,234,285,51
NANODD,CSBS BS,9_months_arm_1,298,257,297,41
NANO,EDI-YC,24_months_arm_1,41,41,41,0
NANO,EDI-YC,36_months_arm_1,13,13,13,0
